In [ ]:
import pandas as pd
import numpy as np
from fancyimpute import  IterativeImputer

In [ ]:
import sys
sys.path.append('../../MyModule/')
from StatAnalyzer import StatAnalyzer
from MissingDataPlotter import MissingDataPlotter
from OutlierRemover import OutlierRemover
from FeaturePlotter import FeaturePlotter
from DataImputer import DataImputer

In [ ]:
train_df = pd.read_csv('./dataset/train.csv')
test_df = pd.read_csv('./dataset/test.csv')
pd.options.display.max_rows = 97
pd.options.display.max_columns = 97

In [ ]:
sa = StatAnalyzer(train_df, 'SalePrice')
mp = MissingDataPlotter(train_df, 'SalePrice')
data_imputer = DataImputer()

In [ ]:
sa.table_diagnose()

In [ ]:
### plot outliers -> kebutuhan analisis
X = train_df.drop(columns=['SalePrice'])
y = train_df['SalePrice']
fp = FeaturePlotter(data=train_df, feature_columns=X.columns)
fp.plot_features(plot_type='boxplot')

In [ ]:
Or = OutlierRemover(train_df)
Or.get_outliers('SalePrice')

In [ ]:
target_col_outlier = ['LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'GrLivArea']
cleaned_df = Or.remove_outliers(target_col_outlier)
sa.table_diagnose(data=cleaned_df)

In [ ]:
fp.plot_features(data=cleaned_df, plot_type='boxplot', showmeans=True)

In [ ]:
mp.missing_plot('heatmap')

In [ ]:
mp.missing_plot('matrix', sort='ascending', label_rotation=90, labels=True)

In [ ]:
mp.plot_missing_data()

In [ ]:
# memasukkan kolom yang mar ke dalam variabel agar mudah dibaca
get_prob_MAR_total_bsmtSF =  train_df.iloc[1, [30, 31, 32, 33, 35]].index.to_list() + ['TotalBsmtSF'] # columns related to -> Total BsmtSF,
get_prob_MAR_garage_cars = train_df.iloc[1, [58, 59, 60, 63, 64]].index.to_list() + ['GarageCars'] # columns related to  -> Garage Cars, Garage Area
get_prob_MAR_fireplaceQu = train_df.iloc[1, [57]].index.to_list() + ['Fireplaces'] # columns related to  -> Fireplaces

print(get_prob_MAR_total_bsmtSF)
print(get_prob_MAR_garage_cars)
print(get_prob_MAR_fireplaceQu)

In [ ]:
target_col = 'SalePrice'
target_isolate_col = 'BsmtQual'
get_isolated_columns = get_prob_MAR_total_bsmtSF

missing_values = train_df[train_df[target_isolate_col].isna()]
complete_values = train_df[~train_df[target_isolate_col].isna()]

display(missing_values.describe(include='all')[get_isolated_columns + [target_col]])
display(complete_values.describe(include='all')[get_isolated_columns + [target_col]])

In [ ]:
target_drop_col = ['Id', 'Alley', 'PoolQC', 'Fence', 'MiscFeature']
copy_df = data_imputer.drop_cols(dataframe=cleaned_df, columns_to_drop=target_drop_col)
copy_df

sebelum membuat dummy variabel atau ohe, pastikan nilai unique dari test dan train itu harus sama jika tidak maka akan ada perbedaan dimensi dan ini akan berpengaruh saat training model

In [ ]:
get_unique_train = sa.table_diagnose(data=train_df, show_dims=False)[['columns', 'types', 'n unique']]
get_unique_train = get_unique_train[get_unique_train['types'] == 'object']

get_unique_cleaned = sa.table_diagnose(data=copy_df, show_dims=False)[['columns', 'types', 'n unique']]
get_unique_cleaned = get_unique_cleaned[get_unique_cleaned['types'] == 'object']

get_unique_test = sa.table_diagnose(data=test_df, show_dims=False)[['columns', 'types', 'n unique']]
get_unique_test = get_unique_test[get_unique_test['types'] == 'object']

display(get_unique_cleaned[['columns', 'n unique']].T)
display(get_unique_train[['columns', 'n unique']].T)
display(get_unique_test[['columns', 'n unique']].T)

menentukan kolom mana saya yang diubah kedalam ordinal dan kolom mana saja yang diubah ke ohe, cari chisquared antara fitur kategori dengan target, gunakan class statanalyzer

In [ ]:
columns_dummies_variable = ['MSZoning', 'Street', 'LotShape', 'LandContour', 'LotConfig', 'LandSlope', 'BldgType']
columns_ordinal = ['Utilities', 'Neighborhood', 'Condition1', 'Condition2', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'SaleType', 'SaleCondition']

copy_df.groupby('GarageQual')['SalePrice'].agg(['mean', 'std', 'count']).sort_values('mean')
# display(copy_df[columns_dummies_variable].nunique())
# display(test_df[columns_dummies_variable].nunique())

In [ ]:
imputed_df = data_imputer.map_and_impute(copy_df, columns_ordinal, columns_dummies_variable, 'SalePrice', 123, 10, 'mean')
display(imputed_df)

In [ ]:
mp.missing_plot(data=imputed_df, plot_type='matrix', sort='ascending', labels=True, label_rotation=90)

In [ ]:
get_weak_corr = sa.concatinated_df(data=imputed_df)
# display(get_weak_corr)

get_weak_corr_cols = get_weak_corr[(get_weak_corr['F'] < 0.2) & (get_weak_corr['p_value'] > 0.05)].index
new_df = data_imputer.drop_cols(dataframe=imputed_df, columns_to_drop=get_weak_corr_cols)

In [ ]:
# fixed_df = data_imputer.create_dummies(new_df, columns=columns_dummies_variable)
# fixed_df

In [ ]:
sa.table_diagnose(new_df)

In [ ]:
import tensorflow as tf
import tensorflow_decision_forests as tfdf
from sklearn.model_selection import train_test_split
from tensorflow.keras import backend as K
from sklearn.model_selection import GridSearchCV
import keras_tuner as kt

print("TensorFlow v" + tf.__version__)
print("TensorFlow Decision Forests v" + tfdf.__version__)

In [ ]:
def split_dataset(dataset, test_ratio=0.20):
  test_indices = np.random.rand(len(dataset)) < test_ratio
  return dataset[~test_indices], dataset[test_indices]

train_ds_pd, valid_ds_pd = split_dataset(new_df)
print("{} examples in training, {} examples in testing.".format(
    len(train_ds_pd), len(valid_ds_pd)))

In [ ]:
label = 'SalePrice'
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(train_ds_pd, label=label, task = tfdf.keras.Task.REGRESSION)
valid_ds = tfdf.keras.pd_dataframe_to_tf_dataset(valid_ds_pd, label=label, task = tfdf.keras.Task.REGRESSION)

In [ ]:
tfdf.keras.get_all_models()

In [ ]:
def r_squared(y_true, y_pred):
    SS_res =  K.sum(K.square(y_true - y_pred)) 
    SS_tot = K.sum(K.square(y_true - K.mean(y_true))) 
    return 1 - SS_res/(SS_tot + K.epsilon())

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

In [ ]:
# model = tfdf.keras.RandomForestModel(hyperparameter_template="benchmark_rank1", task=tfdf.keras.Task.REGRESSION)
model = tfdf.keras.GradientBoostedTreesModel(hyperparameter_template="benchmark_rank1", task = tfdf.keras.Task.REGRESSION)
# # model = tfdf.keras.CartModel(task = tfdf.keras.Task.REGRESSION)

model.compile(metrics=["mse", "mae", r_squared, rmse]) # Optional, you can use this to include a list of eval metrics
model.fit(train_ds)

In [ ]:
tfdf.model_plotter.plot_model_in_colab(model, tree_idx=0, max_depth=3)

inspector = model.make_inspector()
inspector.evaluation()

evaluation = model.evaluate(x=valid_ds,return_dict=True)

for name, value in evaluation.items():
  print(f"{name}: {value:.4f}")

In [ ]:
import matplotlib.pyplot as plt
logs = model.make_inspector().training_logs()
plt.plot([log.num_trees for log in logs], [log.evaluation.rmse for log in logs])
plt.xlabel("Number of trees")
plt.ylabel("RMSE (out-of-bag)")
plt.show()

In [ ]:
# from sklearn.preprocessing import MinMaxScaler
# scaler = MinMaxScaler()

X_train = new_df.drop(columns=['SalePrice'])
y_train = new_df['SalePrice']

# X_scaled = scaler.fit_transform(X)
# X = pd.DataFrame(X_scaled, columns=X.columns)

display(X_train.shape)
display(y_train.shape)

In [ ]:
get_dict = data_imputer.my_dict.copy()
new_test_df = test_df[X_train.columns]

transformed_test_df = new_test_df.replace(get_dict)

In [ ]:
get_obj_df = transformed_test_df.select_dtypes(include='object')
sa.table_diagnose(get_obj_df)

In [ ]:
for col in get_obj_df.columns:
  index = len(get_dict[col])  
  for val in get_obj_df[col].unique():
    if type(val) == str:
      get_obj_df[col] = get_obj_df[col].replace({val:index})
      index += 1

In [ ]:
transformed_test_df[get_obj_df.columns] = get_obj_df

In [ ]:
sa.table_diagnose(transformed_test_df)

In [ ]:
imputer = IterativeImputer(max_iter=10, initial_strategy='mean')
# fit and transform the dataframe values and convert them back to a dataframe
imputed_test_df = pd.DataFrame(imputer.fit_transform(transformed_test_df), columns=transformed_test_df.columns)
# return the imputed dataframe

In [ ]:
sa.table_diagnose(imputed_test_df)

In [ ]:
print(X_train.shape)
print(imputed_test_df.shape)

In [ ]:
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    imputed_test_df,
    task = tfdf.keras.Task.REGRESSION)

y_pred = model.predict(test_ds)

In [ ]:
y_pred

In [ ]:
# Save model predictions to a DataFrame
pred_df = pd.DataFrame(y_pred)

# Import the sample submission (We need the Id column)
sub_df = pd.read_csv('./dataset/sample_submission.csv')


# Concat the predictions DataFrame with the Id column of the sample submission
datasets = pd.concat([sub_df['Id'], pred_df], axis=1)

In [ ]:
# Rename the columns
datasets.columns = ['Id', 'SalePrice']
# Save the predictions to csv file, submission.csv
datasets.to_csv('./dataset/my_submission.csv', index=False)